# ViTreous — `sae.ipynb`

Collect **layer-9 token activations** from the fine-tuned model, train a
**k-sparse autoencoder** (4096 features, k=32), run the **quality gate**,
build the **concept dictionary**, and upload it (ARCHITECTURE.md §9). If
the gate fails, fall back to the k-means provider behind the same
interface. Swap datasets by changing `DATASET`.

In [ ]:
# ─── Knobs ─────────────────────────────────────────────────────────────
DATASET = "eurosat"          # swap datasets with this one string
MODEL   = "vit_s16"
LAYER   = 9                  # SAE probe layer (§9 default)
N_FEATURES = 4096
K = 32
EPOCHS = 50
SEED = 0
DATA_ROOT = "/kaggle/input"
WEIGHTS = None               # fine-tuned .pt from train.ipynb, or None
STORAGE = "supabase"         # "local" for a dry-run
MAX_IMAGES = 2000            # token bank size ≈ MAX_IMAGES * 197

In [ ]:
# Install the vitreous core package (with the [ml] extra) from the repo branch.
# On Kaggle: enable Internet in the notebook settings. The package is the
# single code path shared with the live service — notebooks are thin shells.
import subprocess, sys
REPO = "https://github.com/<owner>/DragonHatchling.git"
BRANCH = "claude/explainable-vit-research-qadg2i"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"git+{REPO}@{BRANCH}#subdirectory=packages/core"], check=False)
# If the repo is attached as a Kaggle dataset/utility script instead, do:
#   %pip install -q -e /kaggle/input/vitreous/packages/core

In [ ]:
import torch, numpy as np
from vitreous.data import get_dataset
from vitreous.models import load_model
from vitreous.instrument import Instrumenter
from vitreous.concepts import (train_sae, quality_gate, build_concept_dictionary,
                               SAEConceptProvider, KMeansConceptProvider,
                               ExemplarRef, layer_token_activations)
from vitreous.storage import get_storage

adapter = get_dataset(DATASET)()
spec = adapter.spec
lm = load_model(MODEL, spec, pretrained=WEIGHTS is None, weights=WEIGHTS,
                num_classes=spec.num_classes)
model = lm.module.eval()

In [ ]:
# Collect layer-9 token activations across the training set (§9).
from PIL import Image
eval_t = adapter.preprocess()
acts, refs = [], []
samples = list(adapter.load(DATA_ROOT, split='train'))[:MAX_IMAGES]
for s in samples:
    img = Image.open(s.image).convert('RGB') if isinstance(s.image, str) else Image.fromarray(s.image)
    x = eval_t(img).unsqueeze(0)
    trace = Instrumenter(model).capture(x)
    toks = layer_token_activations(trace, LAYER)     # [T, D]
    image_id = s.image_id or str(s.image)
    for ti in range(toks.shape[0]):
        acts.append(toks[ti])
        refs.append(ExemplarRef(image_id=image_id, token_idx=ti, activation=0.0, class_label=int(s.label)))
activations = np.stack(acts).astype('float32')
print('token bank:', activations.shape)

In [ ]:
# Train the TopK sparse autoencoder (pure torch).
sae, stats = train_sae(activations, n_features=N_FEATURES, k=K, epochs=EPOCHS, seed=SEED)
print('SAE stats:', stats.to_json())

In [ ]:
# Build the dictionary, then decide SAE-vs-fallback with the quality gate (§9).
provider = SAEConceptProvider(sae, layer=LAYER)
dictionary = build_concept_dictionary(provider, activations, refs,
                                      num_classes=spec.num_classes,
                                      model=MODEL, dataset=DATASET, layer=LAYER)
report = quality_gate(stats, dictionary)
print('quality gate:', report.to_json())

if not report.use_sae:
    print('SAE failed the gate → falling back to k-means:', report.reasons)
    provider = KMeansConceptProvider.fit(activations, n_clusters=512, layer=LAYER, seed=SEED)
    dictionary = build_concept_dictionary(provider, activations, refs,
                                          num_classes=spec.num_classes,
                                          model=MODEL, dataset=DATASET, layer=LAYER)

In [ ]:
# Serialize + upload the dictionary; emit the concept_dictionaries row (§9, §15).
import json, tempfile, os
storage = get_storage(STORAGE)
with tempfile.TemporaryDirectory() as td:
    path = os.path.join(td, f'dictionary_L{LAYER}.json')
    json.dump(dictionary.to_json(), open(path, 'w'))
    key = f'concepts/{MODEL}/L{LAYER}/dictionary.json'
    url = storage.put_file(path, key)
print('dictionary at', url)
print('provider kind:', provider.kind, '| use_sae:', report.use_sae)
# Then insert into concept_dictionaries(model_id, layer, url, quality=report.to_json())
# with the service-role key (see precompute.ipynb for the client pattern).